# Focus Fatigue — Full Pipeline Notebook

Runs Model 1 (Pressure Exposure) → All 4 Signals → Merge into unified dataset.

## How to Use

1. **Set `DATA_ROOT`** in the first code cell below to point to your data.
2. **Optionally set `MATCH_IDS`** to limit which matches to process.
3. **Run all cells** (Cell → Run All).

The notebook will:
- Auto-discover matches from `{DATA_ROOT}/tracking/`
- Run Model 1 (Pressure Exposure) on each match
- Run all 4 registered signals on each match
- Merge everything into a unified parquet file
- Print a summary at the end

---
## ⚙️ INPUT CELL — Configure Your Paths

Edit **`DATA_ROOT`** to point to where your data lives. The expected structure:

```
{DATA_ROOT}/
├── tracking/
│   ├── 2215790/
│   │   └── tracking.parquet
│   ├── 2215791/
│   │   └── tracking.parquet
│   └── ...                (100+ matches)
├── shapes/
│   ├── 2215790.json
│   ├── 2215791.json
│   └── ...
├── shape_outputs/          (legacy V1 shapes, optional)
  ├── team_mappings/
│   └── team_mappings.csv
└── sample/                 (optional 3-match sample)
    ├── 2215790/
    │   └── tracking.parquet
    ├── 2215791/
    │   └── tracking.parquet
    └── 2215792/
        └── tracking.parquet
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THIS CELL — Set your data root and match selection   ║
# ╚══════════════════════════════════════════════════════════════╝

import os
import sys
from pathlib import Path

# ─── POINT THIS TO YOUR DATA DIRECTORY ───
DATA_ROOT = "/mnt/usb/conor_downloads/team_mappings"

# ─── MATCH SELECTION ───
# Leave MATCH_IDS = [] to process ALL matches in the tracking dir.
# Or specify exactly which matches to run:
MATCH_IDS = []  # e.g. ["2215790", "2215791"] or [] for all

# ─── TEST MODE (optional) ───
# Set to a positive number to only load that many frames per match.
# Set to 0 or None to process full matches.
TEST_FRAMES = None  # e.g. 5000 for a quick smoke test

# ─── RESOLVED PATHS (don't edit unless your layout differs) ───
TRACKING_DIR = Path(DATA_ROOT) / "tracking"
SHAPES_DIR = Path(DATA_ROOT) / "shapes"
SAMPLE_DIR = Path(DATA_ROOT) / "sample"
TEAM_MAP_PATH = Path(DATA_ROOT) / "team_mappings" / "team_mappings.csv"

# Fallback: try alternate team map locations
if not TEAM_MAP_PATH.exists():
    TEAM_MAP_PATH = Path(DATA_ROOT) / "team_mappings.csv"
if not TEAM_MAP_PATH.exists():
    TEAM_MAP_PATH = Path(DATA_ROOT).parent / "team_mappings" / "team_mappings.csv"

# Project root
# Robust project root: try notebook path first, fall back to cwd
try:
    # When running via nbconvert, kernel cwd should be repo root
    PROJECT_ROOT = Path.cwd().resolve()
    # Sanity check: if src/ doesn't exist here, try parent
    if not (PROJECT_ROOT / "src").exists():
        # Walk up until we find src/
        for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (parent / "src").exists():
                PROJECT_ROOT = parent
                break
except Exception:
    PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"📂 Data root:    {DATA_ROOT}")
print(f"📂 Tracking:     {TRACKING_DIR}")
print(f"📂 Shapes:       {SHAPES_DIR}")
print(f"📂 Sample:       {SAMPLE_DIR}")
print(f"📄 Team map:     {TEAM_MAP_PATH}  (exists: {TEAM_MAP_PATH.exists()})")
print(f"📂 Project root: {PROJECT_ROOT}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Configure paths in project configs                        ║
# ╚══════════════════════════════════════════════════════════════╝

# Override the default configs with our DATA_ROOT
from src.pressure.config import DEFAULT_CONFIG, PressureConfig
from src.signals.config import DEFAULT_SIGNAL_CONFIG, SignalConfig

# Patch Model 1 paths
DEFAULT_CONFIG.tracking_dir = str(TRACKING_DIR)
DEFAULT_CONFIG.sample_dir = str(SAMPLE_DIR)
DEFAULT_CONFIG.output_dir = str(PROJECT_ROOT / "outputs" / "pressure_exposure")

# Patch signal output path
DEFAULT_SIGNAL_CONFIG.output_root = str(PROJECT_ROOT / "outputs" / "signals")

# Ensure output dirs exist
os.makedirs(DEFAULT_CONFIG.output_dir, exist_ok=True)
os.makedirs(DEFAULT_SIGNAL_CONFIG.output_root, exist_ok=True)
for sig in ["positional_drift", "shift_latency", "pressing_accuracy", "transition_latency"]:
    os.makedirs(os.path.join(DEFAULT_SIGNAL_CONFIG.output_root, sig), exist_ok=True)

print(f"✅ Model 1 output:  {DEFAULT_CONFIG.output_dir}")
print(f"✅ Signals output:  {DEFAULT_SIGNAL_CONFIG.output_root}")
print()

# Also patch the team mappings path in bridge.py for Signal 1
import src.signals.positional_drift.bridge as bridge
bridge._TEAM_MAPPINGS_PATH = TEAM_MAP_PATH
print(f"✅ Team mapping:   {TEAM_MAP_PATH}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Discover matches to process                               ║
# ╚══════════════════════════════════════════════════════════════╝

if MATCH_IDS:
    matches = MATCH_IDS
else:
    tracking_paths = sorted([
        d.name for d in TRACKING_DIR.iterdir()
        if d.is_dir() and (d / "tracking.parquet").exists()
    ])
    
    # If no matches found in tracking/, try sample/
    if not tracking_paths:
        tracking_paths = sorted([
            d.name for d in SAMPLE_DIR.iterdir()
            if d.is_dir() and (d / "tracking.parquet").exists()
        ])
        print("📁 Using sample/ directory")
    
    matches = tracking_paths

if not matches:
    print("❌ No matches found! Check DATA_ROOT and directory structure.")
else:
    print(f"📁 Found {len(matches)} matches to process:")
    for m in matches[:10]:
        print(f"   • {m}")
    if len(matches) > 10:
        print(f"   ... and {len(matches)-10} more")

---
## Phase 1: Model 1 — Pressure Exposure

Computes 4 load indicators per defender per 5-minute block:
- Opponent proximity (opponents within 7m)
- Defensive depth (distance from own goal)
- Reorientation frequency (sharp heading changes)
- Zone transitions (possession changes within 7m)

Combines into a Pressure Composite and classifies blocks as high/low pressure.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Phase 1: Run Model 1 (Pressure Exposure)                  ║
# ╚══════════════════════════════════════════════════════════════╝

import time
import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE 1: MODEL 1 — PRESSURE EXPOSURE")
print("=" * 70)

from src.run_pressure import process_one_match as run_model1

phase1_start = time.time()
model1_results = {}

for idx, match_id in enumerate(matches):
    # Locate tracking data
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        print(f"  ⚠️  [{idx+1}/{len(matches)}] {match_id}: tracking.parquet not found, skipping")
        continue
    
    print(f"\n  [{idx+1}/{len(matches)}] Processing {match_id}...")
    
    if TEST_FRAMES:
        # Quick test: load a small slice and run Model 1 inline
        from src.loaders.load_tracking import load_tracking_statsperform
        from src.smoothing import smooth_trajectory, compute_velocity_features
        from src.segments import split_into_blocks, block_summary
        from src.pressure.opponent_proximity import compute_opponent_proximity, aggregate_opponent_proximity_to_blocks
        from src.pressure.defensive_depth import compute_defensive_depth, aggregate_defensive_depth_to_blocks
        from src.pressure.reorientations import detect_reorientations, aggregate_reorientations_to_blocks
        from src.pressure.transitions import detect_transition_frames, count_zone_transitions, aggregate_transitions_to_blocks
        from src.pressure.composite import build_pressure_dataset, compute_block_baselines, compute_pressure_composite, classify_pressure_blocks
        
        df = load_tracking_statsperform(str(match_path), match_id=match_id)
        df = df.head(TEST_FRAMES)
        df["frame"] = df["frame_count"]
        df = smooth_trajectory(df)
        df = compute_velocity_features(df)
        
        # Compute per-frame metrics BEFORE splitting into blocks
        df = compute_opponent_proximity(df, config=DEFAULT_CONFIG)
        df = compute_defensive_depth(df, config=DEFAULT_CONFIG)
        df = detect_reorientations(df, config=DEFAULT_CONFIG)
        trans_frames = detect_transition_frames(df)
        trans_records = count_zone_transitions(df, trans_frames, config=DEFAULT_CONFIG)
        
        # Now split into blocks and aggregate
        blocks = split_into_blocks(df)
        
        prox_agg = aggregate_opponent_proximity_to_blocks(blocks)
        depth_agg = aggregate_defensive_depth_to_blocks(blocks)
        reo_agg = aggregate_reorientations_to_blocks(blocks)
        trans_agg = aggregate_transitions_to_blocks(blocks, trans_records)
        
        pressure_dataset = build_pressure_dataset(
            blocks, prox_agg, depth_agg, reo_agg, trans_agg,
            config=DEFAULT_CONFIG, game_id=match_id
        )
        
        baselines = compute_block_baselines(pressure_dataset, config=DEFAULT_CONFIG)
        pressure_df = compute_pressure_composite(pressure_dataset, baselines, config=DEFAULT_CONFIG)
        result_df = classify_pressure_blocks(pressure_df, config=DEFAULT_CONFIG)
        
        # Save test outputs
        out_dir = Path(DEFAULT_CONFIG.output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        result_df.to_csv(out_dir / f"pressure_composite_{match_id}.csv", index=False)
        
        n_players = result_df["player_id"].nunique() if "player_id" in result_df.columns else 0
        print(f"  ✅ {match_id}: {n_players} players, {len(result_df)} blocks (TEST MODE)")
    else:
        result = run_model1(match_id, match_path, config=DEFAULT_CONFIG)
        n_players = result.get("n_players", 0)
        elapsed = result.get("elapsed_s", 0)
        print(f"  ✅ {match_id}: {n_players} players in {elapsed:.1f}s")
    
    model1_results[match_id] = result

phase1_elapsed = time.time() - phase1_start
print(f"\n{'=' * 70}")
print(f"Model 1 complete: {len(model1_results)} matches in {phase1_elapsed:.0f}s")
print(f"{'=' * 70}")

---
## Phase 2: All Signals

Runs all 4 registered signals on each match:
- **Signal 1 — Positional Drift:** Distance from expected shape-model position
- **Signal 2 — Shift Latency:** Reaction time to ball speed spikes & opponent runs
- **Signal 3 — Pressing Accuracy:** Fraction of correct presses (Bekkers TTI)
- **Signal 5 — Transition Latency:** Reaction time to possession turnovers

Results saved to `outputs/signals/{signal_name}/{match_id}.csv`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Phase 2: Run all registered signals                       ║
# ╚══════════════════════════════════════════════════════════════╝

print("=" * 70)
print("PHASE 2: ALL SIGNALS")
print("=" * 70)

from src.signals.registry import SIGNAL_REGISTRY, list_signals
from src.signals.base import SignalBase
from src.loaders.load_tracking import load_tracking_statsperform
from src.smoothing import smooth_trajectory, compute_velocity_features
from src.segments import split_into_blocks
from src.signals.output_schema import validate_output

registered = list_signals()
print(f"Registered signals ({len(registered)}): {registered}")
print()

signal_results = {}
phase2_start = time.time()

for idx, match_id in enumerate(matches):
    print(f"\n{'─' * 70}")
    print(f"  Match {idx+1}/{len(matches)}: {match_id}")
    print(f"{'─' * 70}")
    
    # Locate tracking data
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        print(f"  ⚠️  Tracking not found, skipping")
        continue
    
    # Step 1: Load
    print("  [1/4] Loading tracking data...")
    df = load_tracking_statsperform(
        str(match_path), match_id=match_id,
        normalise_dop=True, include_ball=True
    )
    if TEST_FRAMES:
        df = df.head(TEST_FRAMES)
    df["frame"] = df["frame_count"]
    print(f"        {len(df):,} rows, {df['frame_count'].nunique():,} frames")
    
    # Step 2: Smooth
    print("  [2/4] Smoothing trajectories...")
    df = smooth_trajectory(df, inplace=False)
    df = compute_velocity_features(df)
    
    # Step 3: Segment into blocks
    print("  [3/4] Segmenting into blocks...")
    blocks_dfs = split_into_blocks(df)
    print(f"        {len(blocks_dfs)} blocks")
    
    if len(blocks_dfs) == 0:
        print("  ⚠️  No valid blocks, skipping match")
        continue
    
    # Convert blocks to dict format (needed by shift_latency)
    block_dicts = []
    for blk in blocks_dfs:
        bid = str(blk["block_id"].iloc[0])
        phase = int(bid.split("_")[0])
        block_dicts.append({
            "block_id": bid,
            "phase": phase,
            "start_frame": int(blk["frame_count"].min()),
            "end_frame": int(blk["frame_count"].max()),
        })
    
    # Detect teams for pressing_accuracy
    outfield_teams = df[df["player_id"] != -1]["team_id_opta"].unique()
    team_a = int(outfield_teams[0]) if len(outfield_teams) > 0 else 0
    team_b = int(outfield_teams[1]) if len(outfield_teams) > 1 else 0
    
    # Step 4: Run signals
    print("  [4/4] Computing signals...")
    
    match_signal_results = {}
    
    for sig_name in registered:
        t0_sig = time.time()
        print(f"    • {sig_name}...", end=" ", flush=True)
        
        try:
            signal = SIGNAL_REGISTRY[sig_name]()
            
            # Build kwargs for this signal
            kwargs = {"match_df": df, "game_id": match_id}
            
            if sig_name == "shift_latency":
                kwargs["blocks"] = block_dicts
            else:
                kwargs["blocks"] = blocks_dfs
            
            if sig_name == "positional_drift":
                # Find shape file
                shape_paths_to_try = [
                    SHAPES_DIR.parent / "shape_outputs" / f"{match_id}.json",  # V1 legacy
                    SHAPES_DIR / f"{match_id}.json",
                    SAMPLE_DIR / f"{match_id}.json",
                ]
                shape_path = None
                for sp in shape_paths_to_try:
                    if sp.exists():
                        shape_path = sp
                        break
                if shape_path:
                    kwargs["shape_path"] = str(shape_path)
                else:
                    print(f"⚠️ No shape file")
                    match_signal_results[sig_name] = {"rows": 0, "error": "No shape file"}
                    continue
            
            if sig_name == "pressing_accuracy":
                kwargs["own_team_id"] = team_a
                kwargs["opponent_team_id"] = team_b
            
            output_df = signal.compute(**kwargs)
            
            # Validate and save
            signal.validate(output_df)
            signal.save(output_df, match_id=match_id)
            
            n_rows = len(output_df)
            elapsed = time.time() - t0_sig
            print(f"✅ {n_rows} rows in {elapsed:.1f}s")
            match_signal_results[sig_name] = {"rows": n_rows, "elapsed": elapsed}
            
        except Exception as e:
            elapsed = time.time() - t0_sig
            print(f"❌ Error in {elapsed:.1f}s: {e}")
            match_signal_results[sig_name] = {"rows": 0, "error": str(e)}
    
    signal_results[match_id] = match_signal_results

phase2_elapsed = time.time() - phase2_start
print(f"\n{'=' * 70}")
print(f"Signals complete: {len(signal_results)} matches in {phase2_elapsed:.0f}s")
print(f"{'=' * 70}")

---
## Phase 3: Merge into Unified Dataset

Combines Model 1 pressure data with all signal outputs into a single
`unified_fatigue_dataset.parquet` file, joined on `(game_id, block_id, player_id, team_id_opta)`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Phase 3: Merge all outputs                                ║
# ╚══════════════════════════════════════════════════════════════╝

print("=" * 70)
print("PHASE 3: MERGE OUTPUTS")
print("=" * 70)

from src.merge_outputs import merge_all

merge_start = time.time()

unified = merge_all(
    signals_dir=str(PROJECT_ROOT / "outputs" / "signals"),
    pressure_dir=str(PROJECT_ROOT / "outputs" / "pressure_exposure"),
    output_path=str(PROJECT_ROOT / "outputs" / "unified_fatigue_dataset.parquet"),
)

merge_elapsed = time.time() - merge_start
print(f"\nMerge complete in {merge_elapsed:.1f}s")
print(f"Unified dataset: {len(unified)} rows × {len(unified.columns)} columns")

---
## Summary

Final results and next steps.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Pipeline Summary                                         ║
# ╚══════════════════════════════════════════════════════════════╝

total_elapsed = time.time() - phase1_start

print("=" * 70)
print("PIPELINE COMPLETE — SUMMARY")
print("=" * 70)
print()
print(f"  Matches attempted: {len(matches)}")
print(f"  Model 1 success:   {len(model1_results)}")
print(f"  Signals success:   {len(signal_results)}")
print(f"  Total time:        {total_elapsed:.0f}s ({total_elapsed/60:.1f}min)")
print()

# Per-signal summary across all matches
if signal_results:
    print(f"  {'Signal':<25} {'Rows':>8} {'Errors':>8}")
    print(f"  {'─' * 41}")
    for sig_name in registered:
        total_rows = sum(
            r[sig_name]["rows"]
            for r in signal_results.values()
            if sig_name in r
        )
        errors = [
            mid for mid, r in signal_results.items()
            if sig_name in r and r[sig_name].get("error")
        ]
        err_str = f"{len(errors)}" if errors else "─"
        print(f"  {sig_name:<25} {total_rows:>8,} {err_str:>8}")

print()
print(f"  Output files:")
print(f"    Pressure:  {DEFAULT_CONFIG.output_dir}/")
print(f"    Signals:   {DEFAULT_SIGNAL_CONFIG.output_root}/{{signal}}/{{match}}.csv")
print(f"    Unified:   {PROJECT_ROOT / 'outputs' / 'unified_fatigue_dataset.parquet'}")
print()
print(f"  Next steps:")
print(f"    1. Inspect outputs for sanity")
print(f"    2. Run analysis on the unified parquet")
print(f"    3. Check audit_report.md for known issues")